# 01 - Data Download (Binance Monthly ZIPs)

Implements spec Section 4 and checkpoint 8A.1.

Outputs:
- `data/raw_data/zips/` (monthly ZIPs + `.CHECKSUM`)
- `data/raw_data/download_manifest.json`
- `data/raw_data/validation_report.json`
- `data/raw_data/klines_1m.parquet`


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from tqdm import tqdm

# Resolve repo root robustly (works whether you launch Jupyter from repo root or notebooks/)
ROOT = Path.cwd()
if not (ROOT / 'docs' / 'MINIMAL_PROJECT_SPEC_v2.md').exists():
    if (ROOT.parent / 'docs' / 'MINIMAL_PROJECT_SPEC_v2.md').exists():
        ROOT = ROOT.parent
    else:
        raise RuntimeError('Could not locate repo root (docs/MINIMAL_PROJECT_SPEC_v2.md not found).')

sys.path.insert(0, str(ROOT))
from src import utils

RAW_DIR = ROOT / 'data' / 'raw_data'
ZIPS_DIR = RAW_DIR / 'zips'
ZIPS_DIR.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)
print('ZIPS_DIR:', ZIPS_DIR)


ROOT: c:\Users\vitil\OneDrive\Desktop\barrier_classifier
ZIPS_DIR: c:\Users\vitil\OneDrive\Desktop\barrier_classifier\data\raw_data\zips


In [2]:
# Generate URL list (spec Section 4.2)
config = {
    'SYMBOL': utils.SYMBOL,
    'INTERVAL': utils.INTERVAL,
    'START_YEAR': utils.START_YEAR,
    'START_MONTH': utils.START_MONTH,
    'END_YEAR': utils.END_YEAR,
    'END_MONTH': utils.END_MONTH,
}

url_entries = utils.generate_download_urls(
    symbol=utils.SYMBOL,
    interval=utils.INTERVAL,
    start_year=utils.START_YEAR,
    start_month=utils.START_MONTH,
    end_year=utils.END_YEAR,
    end_month=utils.END_MONTH,
)

print('Months to fetch:', len(url_entries))
display(pd.DataFrame(url_entries).head())


Months to fetch: 12


,year,month,data_url,checksum_url,filename
0,2025,1,https://data.binance.vision/data/spot/monthly/...,https://data.binance.vision/data/spot/monthly/...,BTCUSDT-1m-2025-01.zip
1,2025,2,https://data.binance.vision/data/spot/monthly/...,https://data.binance.vision/data/spot/monthly/...,BTCUSDT-1m-2025-02.zip
2,2025,3,https://data.binance.vision/data/spot/monthly/...,https://data.binance.vision/data/spot/monthly/...,BTCUSDT-1m-2025-03.zip
3,2025,4,https://data.binance.vision/data/spot/monthly/...,https://data.binance.vision/data/spot/monthly/...,BTCUSDT-1m-2025-04.zip
4,2025,5,https://data.binance.vision/data/spot/monthly/...,https://data.binance.vision/data/spot/monthly/...,BTCUSDT-1m-2025-05.zip


In [3]:
# Download ZIP + checksum for each month; verify checksum (spec Section 4.2)
downloads = []

for entry in tqdm(url_entries, desc='Downloading monthly ZIPs'):
    filename = entry['filename']
    zip_path = ZIPS_DIR / filename
    checksum_path = ZIPS_DIR / (filename + '.CHECKSUM')

    record = {
        **entry,
        'zip_path': str(zip_path),
        'checksum_path': str(checksum_path),
        'status': 'pending',
    }

    try:
        if zip_path.exists() and checksum_path.exists() and utils.verify_checksum(zip_path, checksum_path):
            record['status'] = 'ok_cached'
            downloads.append(record)
            continue

        ok_checksum = utils.download_file(entry['checksum_url'], checksum_path)
        ok_zip = utils.download_file(entry['data_url'], zip_path)

        if not ok_checksum or not ok_zip:
            record['status'] = 'failed'
            record['error'] = 'download_failed'
            downloads.append(record)
            continue

        if not utils.verify_checksum(zip_path, checksum_path):
            record['status'] = 'failed'
            record['error'] = 'checksum_mismatch'
            # Fail-fast: remove corrupted zip to force clean retry on rerun
            try:
                zip_path.unlink(missing_ok=True)
            except Exception:
                pass
            downloads.append(record)
            continue

        record['status'] = 'ok_downloaded'
        downloads.append(record)

    except Exception as e:
        record['status'] = 'failed'
        record['error'] = f'{type(e).__name__}: {e}'
        downloads.append(record)

manifest = {
    'config': config,
    'downloads': downloads,
}
utils.save_json(RAW_DIR / 'download_manifest.json', manifest)

failed = [d for d in downloads if d['status'] not in ('ok_cached', 'ok_downloaded')]
print('Failed months:', len(failed))
if failed:
    display(pd.DataFrame(failed))
    raise RuntimeError('One or more monthly downloads failed; see data/raw_data/download_manifest.json')

print('All downloads OK')


Failed months: 0
All downloads OK


In [4]:
# Extract CSVs, concatenate, convert timestamps, validate, and write parquet (spec Section 4.2-4.5)
dfs = []
for d in tqdm(downloads, desc='Extracting CSVs'):
    if d['status'] not in ('ok_cached', 'ok_downloaded'):
        continue
    zip_path = Path(d['zip_path'])
    df_m = utils.extract_and_load_csv(zip_path)
    # Drop unused field early to keep dataset clean
    if 'ignore' in df_m.columns:
        df_m = df_m.drop(columns=['ignore'])
    dfs.append(df_m)

raw = pd.concat(dfs, ignore_index=True)
raw = raw.sort_values('open_time').reset_index(drop=True)
df = utils.convert_timestamps(raw)
df = df.sort_index()

# Enforce a complete 1-minute grid (spec requires no gaps).
expected_start_open = pd.Timestamp(
    f"{config['START_YEAR']}-{config['START_MONTH']:02d}-01",
    tz='UTC',
)
expected_end_open_exclusive = (
    pd.Timestamp(f"{config['END_YEAR']}-{config['END_MONTH']:02d}-01", tz='UTC')
    + pd.offsets.MonthBegin(1)
)
expected_start_ts = expected_start_open + pd.Timedelta(minutes=1)
expected_end_ts = expected_end_open_exclusive
expected_index = pd.date_range(start=expected_start_ts, end=expected_end_ts, freq='1min', tz='UTC')

df = df.reindex(expected_index)
missing = df['close'].isna()
n_missing = int(missing.sum())
if n_missing:
    close_ffill = df['close'].ffill()
    if close_ffill.isna().any():
        raise RuntimeError('Cannot fill missing bars: leading NaNs after ffill')
    for col in ['open', 'high', 'low', 'close']:
        df.loc[missing, col] = close_ffill[missing]
    for col in ['volume', 'quote_volume', 'taker_buy_base', 'taker_buy_quote']:
        if col in df.columns:
            df.loc[missing, col] = 0.0
    if 'num_trades' in df.columns:
        df.loc[missing, 'num_trades'] = 0
print('Filled missing bars:', n_missing)

# Validation report (spec Section 12.4 schema)
report = utils.validate_klines(df)
validation_report = {
    'is_valid': bool(report.get('is_valid', False)),
    'n_rows': int(report.get('n_rows', len(df))),
    'date_range': [str(df.index.min()), str(df.index.max())],
    'issues': report.get('issues', []),
}
if 'gap_locations' in report:
    validation_report['gap_locations'] = report['gap_locations']
utils.save_json(RAW_DIR / 'validation_report.json', validation_report)

# Fail fast on any remaining validation issues.
issues = report.get('issues', [])
if issues:
    raise RuntimeError(f"Raw data validation failed: {issues}")

# Checkpoint 8A.1 (critical)
utils.checkpoint_raw_data(df, config={
    **config,
    'START_YEAR': utils.START_YEAR,
    'START_MONTH': utils.START_MONTH,
    'END_YEAR': utils.END_YEAR,
    'END_MONTH': utils.END_MONTH,
})

out_path = RAW_DIR / 'klines_1m.parquet'
df.to_parquet(out_path)
print('Wrote:', out_path)
print('Rows:', len(df))
print('Range:', df.index.min(), 'to', df.index.max())


Extracting CSVs: 100%|██████████| 12/12 [00:00<00:00, 14.38it/s]


Filled missing bars: 0
OK: Raw data validation passed
Wrote: c:\Users\vitil\OneDrive\Desktop\barrier_classifier\data\raw_data\klines_1m.parquet
Rows: 525600
Range: 2025-01-01 00:01:00+00:00 to 2026-01-01 00:00:00+00:00


In [5]:
# ============================================================
# DERIVATIVES DATA DOWNLOAD (Appendix E)
# ============================================================

if utils.ENABLE_DERIVATIVES_FEATURES:
    DERIV_DIR = ROOT / utils.DERIVATIVES_RAW_DIR
    DERIV_ZIPS_DIR = DERIV_DIR / 'zips'
    DERIV_DIR.mkdir(parents=True, exist_ok=True)
    DERIV_ZIPS_DIR.mkdir(parents=True, exist_ok=True)

    deriv_urls = utils.generate_all_derivatives_urls(
        start_year=utils.START_YEAR,
        start_month=utils.START_MONTH,
        end_year=utils.END_YEAR,
        end_month=utils.END_MONTH,
    )

    def _download_entries(entries, label):
        records = []
        for entry in tqdm(entries, desc=f'Downloading {label}'):
            filename = entry['filename']
            zip_path = DERIV_ZIPS_DIR / filename
            checksum_path = DERIV_ZIPS_DIR / (filename + '.CHECKSUM')

            record = {
                **entry,
                'zip_path': str(zip_path),
                'checksum_path': str(checksum_path),
                'status': 'pending',
            }

            try:
                if zip_path.exists() and checksum_path.exists() and utils.verify_checksum(zip_path, checksum_path):
                    record['status'] = 'ok_cached'
                    records.append(record)
                    continue

                ok_checksum = utils.download_file(entry['checksum_url'], checksum_path)
                ok_zip = utils.download_file(entry['data_url'], zip_path)

                if not ok_checksum or not ok_zip:
                    record['status'] = 'failed'
                    record['error'] = 'download_failed'
                    records.append(record)
                    continue

                if not utils.verify_checksum(zip_path, checksum_path):
                    record['status'] = 'failed'
                    record['error'] = 'checksum_mismatch'
                    try:
                        zip_path.unlink(missing_ok=True)
                    except Exception:
                        pass
                    records.append(record)
                    continue

                record['status'] = 'ok_downloaded'
                records.append(record)

            except Exception as e:
                record['status'] = 'failed'
                record['error'] = str(e)
                records.append(record)
        return records

    def _load_concat(records, loader, label):
        frames = []
        for rec in tqdm(records, desc=f'Loading {label}'):
            if rec['status'] not in ('ok_cached', 'ok_downloaded'):
                continue
            frames.append(loader(Path(rec['zip_path'])))
        if not frames:
            print(f'No files loaded for {label}')
            return pd.DataFrame()
        out = pd.concat(frames)
        out = out.sort_index()
        out = out[~out.index.duplicated(keep='last')]
        return out

    downloads = {}
    if utils.ENABLE_FUTURES_KLINES:
        downloads['futures_klines'] = _download_entries(deriv_urls['futures_klines'], 'futures klines')
    else:
        downloads['futures_klines'] = []

    if utils.ENABLE_FUNDING_RATE:
        downloads['funding_rate'] = _download_entries(deriv_urls['funding_rate'], 'funding rate')
    else:
        downloads['funding_rate'] = []

    if utils.ENABLE_FUTURES_METRICS:
        downloads['futures_metrics'] = _download_entries(deriv_urls['futures_metrics'], 'futures metrics')
    else:
        downloads['futures_metrics'] = []

    if utils.ENABLE_EOH_SUMMARY:
        downloads['eoh_summary'] = _download_entries(deriv_urls['eoh_summary'], 'EOH summary')
    else:
        downloads['eoh_summary'] = []

    if utils.ENABLE_BVOL_INDEX:
        downloads['bvol_index'] = _download_entries(deriv_urls['bvol_index'], 'BVOL index')
    else:
        downloads['bvol_index'] = []

    index_1m = df.index

    if utils.ENABLE_FUTURES_KLINES:
        fut_raw = _load_concat(downloads['futures_klines'], utils.load_futures_klines, 'futures klines')
        if fut_raw.empty:
            df_fut = pd.DataFrame(
                index=index_1m,
                columns=[
                    'open', 'high', 'low', 'close', 'volume', 'quote_volume',
                    'num_trades', 'taker_buy_base', 'taker_buy_quote',
                ],
            )
        else:
            df_fut = utils.align_to_1m_grid(fut_raw, index_1m, method=None)
        df_fut.to_parquet(DERIV_DIR / 'futures_klines_1m.parquet')
        print('Wrote futures klines:', len(df_fut))
    else:
        df_fut = pd.DataFrame(index=index_1m)

    if utils.ENABLE_FUNDING_RATE:
        funding_raw = _load_concat(downloads['funding_rate'], utils.load_funding_rate, 'funding rate')
        if funding_raw.empty:
            df_funding = pd.DataFrame(index=index_1m, columns=['funding_rate'])
        else:
            df_funding = utils.align_to_1m_grid(funding_raw, index_1m, method='ffill')
        df_funding.to_parquet(DERIV_DIR / 'funding_rate_1m.parquet')
        print('Wrote funding rate:', len(df_funding))
    else:
        df_funding = pd.DataFrame(index=index_1m)

    if utils.ENABLE_FUTURES_METRICS:
        metrics_raw = _load_concat(downloads['futures_metrics'], utils.load_futures_metrics, 'futures metrics')
        if metrics_raw.empty:
            df_metrics = pd.DataFrame(index=index_1m, columns=['oi_usd'])
        else:
            df_metrics = utils.align_to_1m_grid(metrics_raw, index_1m, method='ffill')
        df_metrics.to_parquet(DERIV_DIR / 'futures_metrics_1m.parquet')
        print('Wrote futures metrics:', len(df_metrics))
    else:
        df_metrics = pd.DataFrame(index=index_1m)

    if utils.ENABLE_EOH_SUMMARY:
        eoh_raw = _load_concat(downloads['eoh_summary'], utils.load_eoh_summary, 'EOH summary')
        if eoh_raw.empty:
            df_eoh = pd.DataFrame(
                index=index_1m,
                columns=[
                    'opt_oi', 'put_open_interest', 'call_open_interest',
                    'opt_volume', 'put_volume', 'call_volume',
                ],
            )
        else:
            df_eoh = utils.align_to_1m_grid(eoh_raw, index_1m, method='ffill')
        df_eoh.to_parquet(DERIV_DIR / 'eoh_summary_1m.parquet')
        print('Wrote EOH summary:', len(df_eoh))
    else:
        df_eoh = pd.DataFrame(index=index_1m)

    if utils.ENABLE_BVOL_INDEX:
        bvol_raw = _load_concat(downloads['bvol_index'], utils.load_bvol_index, 'BVOL index')
        if bvol_raw.empty:
            df_bvol = pd.DataFrame(index=index_1m, columns=['bvol'])
        else:
            df_bvol = utils.align_to_1m_grid(bvol_raw, index_1m, method='ffill')
        df_bvol.to_parquet(DERIV_DIR / 'bvol_index_1m.parquet')
        print('Wrote BVOL index:', len(df_bvol))
    else:
        df_bvol = pd.DataFrame(index=index_1m)

    if (
        utils.ENABLE_FUTURES_KLINES
        and utils.ENABLE_FUNDING_RATE
        and utils.ENABLE_FUTURES_METRICS
        and utils.ENABLE_EOH_SUMMARY
        and utils.ENABLE_BVOL_INDEX
    ):
        deriv_report = utils.checkpoint_derivatives_data(df_fut, df_funding, df_metrics, df_eoh, df_bvol, df)
        utils.save_json(DERIV_DIR / 'derivatives_validation.json', deriv_report)
        print('Wrote:', DERIV_DIR / 'derivatives_validation.json')


Loading futures klines: 100%|██████████| 12/12 [00:00<00:00, 17.87it/s]


Wrote futures klines: 525600


Loading funding rate: 100%|██████████| 12/12 [00:00<00:00, 501.70it/s]


Wrote funding rate: 525600


Loading futures metrics: 100%|██████████| 365/365 [00:00<00:00, 398.07it/s]


Wrote futures metrics: 525600


Loading EOH summary: 0it [00:00, ?it/s]


No files loaded for EOH summary
Wrote EOH summary: 525600


Loading BVOL index: 100%|██████████| 365/365 [00:13<00:00, 26.48it/s]


Wrote BVOL index: 525600
OK: Derivatives data validation passed
Wrote: c:\Users\vitil\OneDrive\Desktop\barrier_classifier\data\raw_data\derivatives\derivatives_validation.json
